# 06 Complete Pre-LN Transformer Block & Autograd Gradient Flow

Building a PyTorch Pre-LN Transformer block with causal masking and simulating 30-layer backpropagation gradient flow.

## Part 1: PyTorch Pre-LN Transformer Block Module

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.style.use('dark_background')

class PreLNTransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        
    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.ffn(self.norm2(x))
        return x

block = PreLNTransformerBlock(d_model=256, n_heads=8, d_ff=1024)
x = torch.randn(2, 16, 256)
out = block(x)
print('Pre-LN Transformer Block PyTorch Output Shape:', out.shape)

## Part 2: 30-Layer PyTorch Autograd Backpropagation Simulation (Pre-LN vs Post-LN)
Tracking activation gradient norms $(\|\nabla_{x_l} \mathcal{L}\|)$ backpropagating from Layer 30 to Layer 1.

In [ ]:
class PostLNLayer(nn.Module):
    def __init__(self, d=64):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.linear = nn.Linear(d, d)
    def forward(self, x): return self.norm(x + F.relu(self.linear(x)))

class PreLNLayer(nn.Module):
    def __init__(self, d=64):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.linear = nn.Linear(d, d)
    def forward(self, x): return x + self.linear(F.relu(self.norm(x)))

num_layers = 30
post_blocks = nn.ModuleList([PostLNLayer() for _ in range(num_layers)])
pre_blocks = nn.ModuleList([PreLNLayer() for _ in range(num_layers)])

x_post = torch.randn(1, 5, 64, requires_grad=True)
h = x_post
post_acts = [h]
for b in post_blocks:
    h = b(h); h.retain_grad(); post_acts.append(h)
h.sum().backward()
post_grads = [a.grad.norm().item() for a in post_acts[1:]]

x_pre = torch.randn(1, 5, 64, requires_grad=True)
h = x_pre
pre_acts = [h]
for b in pre_blocks:
    h = b(h); h.retain_grad(); pre_acts.append(h)
h.sum().backward()
pre_grads = [a.grad.norm().item() for a in pre_acts[1:]]

plt.figure(figsize=(9, 4.5))
plt.plot(range(1, 31), post_grads, marker='o', color='#ff6b6b', label='Post-LN (Exponential Gradient Decay)')
plt.plot(range(1, 31), pre_grads, marker='s', color='#51cf66', label='Pre-LN (Uniform Residual Highway)')
plt.title('PyTorch Autograd Gradient Norm ||dL/dx_l|| across 30 Transformer Layers')
plt.xlabel('Layer Index (1=Input, 30=Output)'); plt.ylabel('Activation Gradient Norm')
plt.grid(True); plt.legend(); plt.show()